# Beginner 01 — Agent Identity Foundations

**Enterprise scenario:** an employee asks a procurement agent to purchase project supplies.

This lab exposes the security mechanics instead of hiding them behind an agent framework.

### You will
- model first-class principals;
- preserve requester, actor and workload identities;
- demonstrate an unsafe tool call;
- add deterministic authorization;
- create bounded delegation;
- reproduce a confused-deputy problem;
- generate audit evidence;
- run negative/adversarial policy tests.


In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from enum import Enum
from typing import Optional
import json, uuid

def utcnow():
    return datetime.now(timezone.utc)

print("Lab ready:", utcnow().isoformat())


## 1 — Model first-class principals

A human, logical agent, workload, and service are separate security principals.

In [ ]:
class PrincipalType(str, Enum):
    USER = "user"
    AGENT = "agent"
    WORKLOAD = "workload"
    SERVICE = "service"

@dataclass(frozen=True)
class Principal:
    id: str
    type: PrincipalType
    display_name: str

alice = Principal("user:alice", PrincipalType.USER, "Alice")
procurement_agent = Principal("agent:procurement", PrincipalType.AGENT, "Procurement Agent")
prod_workload = Principal("spiffe://example.com/prod/procurement", PrincipalType.WORKLOAD, "Production workload")
purchasing_api = Principal("service:purchasing-api", PrincipalType.SERVICE, "Purchasing API")

for principal in [alice, procurement_agent, prod_workload, purchasing_api]:
    print(principal)


## 2 — Explicit identity context

Security context travels through trusted application plumbing; the LLM must not invent it.

In [ ]:
@dataclass(frozen=True)
class IdentityContext:
    requester: Principal
    actor: Principal
    workload: Optional[Principal]
    task_id: str
    purpose: str
    delegation_chain: tuple[str, ...] = ()

ctx = IdentityContext(
    requester=alice,
    actor=procurement_agent,
    workload=prod_workload,
    task_id=f"task:{uuid.uuid4().hex[:8]}",
    purpose="purchase project supplies",
)
ctx


## 3 — Unsafe tool

This function performs an action but never decides whether the caller is entitled to it.

In [ ]:
PURCHASES = []

def unsafe_create_purchase_order(*, sku: str, amount: float):
    order = {"id": f"po:{len(PURCHASES)+1}", "sku": sku, "amount": amount}
    PURCHASES.append(order)
    return order

unsafe_create_purchase_order(sku="GPU-001", amount=25_000)


A system prompt saying *never spend above $500* would not make the protected operation safe. Enforcement belongs in trusted code.

## 4 — Deterministic policy decision

In [ ]:
@dataclass(frozen=True)
class AuthorizationRequest:
    requester_id: str
    actor_id: str
    workload_id: Optional[str]
    action: str
    resource: str
    amount: float
    purpose: str

@dataclass(frozen=True)
class Decision:
    allowed: bool
    reason: str

def decide(req: AuthorizationRequest) -> Decision:
    if req.actor_id != "agent:procurement":
        return Decision(False, "Only procurement agent may create purchase orders")
    if req.workload_id != "spiffe://example.com/prod/procurement":
        return Decision(False, "Only production workload may perform this action")
    if req.action != "purchase:create":
        return Decision(False, "Unsupported action")
    if req.amount > 500:
        return Decision(False, "Agent purchase limit is CAD 500")
    if not req.resource.startswith("project:atlas/"):
        return Decision(False, "Authority restricted to Project Atlas")
    return Decision(True, "Policy requirements satisfied")

def authorize(ctx, *, action, resource, amount):
    return decide(AuthorizationRequest(
        requester_id=ctx.requester.id,
        actor_id=ctx.actor.id,
        workload_id=ctx.workload.id if ctx.workload else None,
        action=action, resource=resource, amount=amount, purpose=ctx.purpose,
    ))

for resource, amount in [
    ("project:atlas/purchase-order", 120),
    ("project:atlas/purchase-order", 900),
    ("project:secret/purchase-order", 50),
]:
    print(resource, amount, authorize(ctx, action="purchase:create", resource=resource, amount=amount))


## 5 — Policy enforcement in front of the tool

In [ ]:
AUDIT_LOG = []

def audit(ctx, action, resource, amount, decision):
    event = {
        "timestamp": utcnow().isoformat(),
        "requester": ctx.requester.id,
        "actor": ctx.actor.id,
        "workload": ctx.workload.id if ctx.workload else None,
        "task_id": ctx.task_id,
        "purpose": ctx.purpose,
        "delegation_chain": list(ctx.delegation_chain),
        "action": action,
        "resource": resource,
        "amount": amount,
        "decision": "allow" if decision.allowed else "deny",
        "reason": decision.reason,
    }
    AUDIT_LOG.append(event)
    return event

def create_purchase_order(ctx, *, project, sku, amount):
    resource = f"project:{project}/purchase-order"
    decision = authorize(ctx, action="purchase:create", resource=resource, amount=amount)
    audit(ctx, "purchase:create", resource, amount, decision)
    if not decision.allowed:
        raise PermissionError(decision.reason)
    return unsafe_create_purchase_order(sku=sku, amount=amount)

print(create_purchase_order(ctx, project="atlas", sku="CHAIR-01", amount=180))
try:
    create_purchase_order(ctx, project="atlas", sku="SERVER-99", amount=4_000)
except PermissionError as e:
    print("DENIED:", e)

print(json.dumps(AUDIT_LOG[-1], indent=2))


## 6 — Bounded delegation

Alice's authority is not copied. A narrower grant is issued to the agent.

In [ ]:
@dataclass(frozen=True)
class Delegation:
    id: str
    delegator: str
    delegate: str
    action: str
    resource_prefix: str
    max_amount: float
    expires_at: datetime

delegation = Delegation(
    id=f"dlg:{uuid.uuid4().hex[:8]}",
    delegator=alice.id,
    delegate=procurement_agent.id,
    action="purchase:create",
    resource_prefix="project:atlas/",
    max_amount=500,
    expires_at=utcnow() + timedelta(hours=1),
)

def validate_delegation(d, ctx, *, action, resource, amount):
    if d.delegator != ctx.requester.id:
        return Decision(False, "Requester is not delegator")
    if d.delegate != ctx.actor.id:
        return Decision(False, "Actor is not delegate")
    if d.action != action:
        return Decision(False, "Action was not delegated")
    if not resource.startswith(d.resource_prefix):
        return Decision(False, "Resource outside delegated scope")
    if amount > d.max_amount:
        return Decision(False, "Amount exceeds delegated authority")
    if utcnow() >= d.expires_at:
        return Decision(False, "Delegation expired")
    return Decision(True, "Delegation valid")

for amount in [100, 700]:
    print(amount, validate_delegation(
        delegation, ctx,
        action="purchase:create",
        resource="project:atlas/purchase-order",
        amount=amount,
    ))


## 7 — Confused deputy attack

The repository recognizes a privileged agent but ignores the requesting human.

In [ ]:
DOCUMENTS = {
    "project-atlas-plan": {"classification": "project", "content": "Atlas launch plan"},
    "executive-payroll": {"classification": "payroll", "content": "Highly confidential payroll"},
}

def unsafe_document_read(agent_id, document_id):
    if not agent_id.startswith("agent:"):
        raise PermissionError("not an agent")
    return DOCUMENTS[document_id]["content"]

print("Unsafe result:", unsafe_document_read(procurement_agent.id, "executive-payroll"))


### Fix: preserve requester identity and authorize the resource

In [ ]:
USER_DOC_ACCESS = {
    "user:alice": {"project-atlas-plan"},
    "user:bob": set(),
}

def safe_document_read(ctx, document_id):
    if ctx.actor.id != "agent:procurement":
        raise PermissionError("unexpected actor")
    if document_id not in USER_DOC_ACCESS.get(ctx.requester.id, set()):
        raise PermissionError(f"{ctx.requester.id} is not authorized for document:{document_id}")
    return DOCUMENTS[document_id]["content"]

print(safe_document_read(ctx, "project-atlas-plan"))
try:
    safe_document_read(ctx, "executive-payroll")
except PermissionError as e:
    print("DENIED:", e)


## 8 — Adversarial authorization tests

In [ ]:
research_agent = Principal("agent:research", PrincipalType.AGENT, "Research Agent")
dev_workload = Principal("spiffe://example.com/dev/research", PrincipalType.WORKLOAD, "Development workload")

def make_ctx(actor=procurement_agent, workload=prod_workload):
    return IdentityContext(
        requester=alice, actor=actor, workload=workload,
        task_id=f"task:{uuid.uuid4().hex[:8]}",
        purpose="purchase project supplies",
    )

cases = [
    ("valid", make_ctx(), "project:atlas/purchase-order", 100, True),
    ("too expensive", make_ctx(), "project:atlas/purchase-order", 501, False),
    ("wrong resource", make_ctx(), "project:secret/purchase-order", 10, False),
    ("wrong agent", make_ctx(actor=research_agent), "project:atlas/purchase-order", 10, False),
    ("dev workload", make_ctx(workload=dev_workload), "project:atlas/purchase-order", 10, False),
]

for name, test_ctx, resource, amount, expected in cases:
    d = authorize(test_ctx, action="purchase:create", resource=resource, amount=amount)
    assert d.allowed is expected, (name, d)
    print(f"{name:15} {'ALLOW' if d.allowed else 'DENY':5}  {d.reason}")


## 9 — Challenge: delegate to a sub-agent

Create `agent:vendor-research` with authority to search vendors for Project Atlas but **not** create purchase orders.

Requirements:

- 10-minute expiry;
- separate narrower delegation;
- delegation ID added to the actor chain;
- negative test proving it cannot purchase.

Preserve:

```text
authority(sub-agent) ⊆ authority(procurement-agent) ⊆ authority(user)
```


In [ ]:
vendor_research_agent = Principal(
    "agent:vendor-research",
    PrincipalType.AGENT,
    "Vendor Research Agent",
)

# Implement the narrower delegation and tests here.


## 10 — Review questions

1. Why is an agent identifier not an authentication mechanism?
2. Why should logical agent identity and workload identity both be preserved?
3. What is lost when an agent simply forwards the user's credential?
4. Where should the policy enforcement point sit relative to the LLM and tool?
5. Why should delegation normally attenuate authority?
6. How does preserving requester identity mitigate a confused-deputy problem?
7. Which fields would you need to investigate a suspicious purchase six months later?

## Next course
**Beginner 02 — Humans, Workloads and Agents**
